# Base Colab Notebook

## 0. Runtime Setup
### 0.1 Configure Base Runtime Inputs

What this step does: defines the repository revision, fixed Colab roots, Base run name, run mode, and output conflict policy used by later workflow-owned planning.

Required inputs: operator-edited REPO_URL, REPO_REF, RUN_MODE, RUN_NAME, and BASE_OUTPUT_POLICY values. WORKTREE_ROOT and DRIVE_PROJECT_ROOT are fixed Colab project roots and should only change if the project-wide Colab layout changes.

Produced outputs: configured constants printed for review before Drive, repository, or artifact restore operations run.

When this step may be skipped: only when these constants are already defined in the current runtime exactly as intended for this Base run.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
WORKTREE_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")

# smoke -> tiny quick pipeline run
# check -> full data, 1 epoch, not final training
# complete -> final M0 baseline training intent
RUN_MODE = "smoke"  # "smoke", "check", or "complete"
RUN_NAME = "smoke_run"
BASE_OUTPUT_POLICY = "fail"

print("Worktree root:", WORKTREE_ROOT)
print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Run mode:", RUN_MODE)
print("Run name:", RUN_NAME)
print("Output policy:", BASE_OUTPUT_POLICY)


### 0.2 Mount Google Drive

What this step does: mounts Google Drive at /content/drive and confirms whether the mirrored project root is already present under MyDrive.

Required inputs: DRIVE_PROJECT_ROOT from step 0.1 and an authenticated Colab Drive session.

Produced outputs: a mounted Drive filesystem and an existence check for DRIVE_PROJECT_ROOT.

When this step may be skipped: only when Drive is already mounted and DRIVE_PROJECT_ROOT has been checked in this runtime.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print("Drive project root exists:", DRIVE_PROJECT_ROOT.is_dir())


### 0.3 Ensure zstd Is Available

What this step does: checks for the zstd command and installs the Debian package when the Colab runtime does not provide it.

Required inputs: Colab apt package access and sudo permission in the runtime.

Produced outputs: a runtime where zstd --version succeeds before any workflow archive command is used.

When this step may be skipped: only when shutil.which("zstd") already finds zstd and zstd --version has succeeded in this runtime.

In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")

zstd_path = shutil.which("zstd")
if zstd_path is None:
    raise RuntimeError("zstd path was not resolved after preflight.")
print("zstd path:", zstd_path)

!zstd --version

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")


### 0.4 Acquire Repository

What this step does: creates the runtime checkout when missing, fetches repository refs, checks out REPO_REF, and prints the resolved branch and revision.

Required inputs: REPO_URL, REPO_REF, writable /content, and network access to the repository.

Produced outputs: a repository checkout at WORKTREE_ROOT with the requested revision checked out.

When this step may be skipped: only when WORKTREE_ROOT already contains the requested revision and the operator intentionally wants to reuse it.

In [ ]:
%cd /content/

if WORKTREE_ROOT.exists():
    print("Repository checkout already exists:", WORKTREE_ROOT)
else:
    !git clone {REPO_URL} {WORKTREE_ROOT}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to clone repository.")

!git -C {WORKTREE_ROOT} fetch --all --tags

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to fetch repository refs.")

!git -C {WORKTREE_ROOT} checkout {REPO_REF}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print("Repository branch:")
!git -C {WORKTREE_ROOT} branch --show-current

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out branch.")

print("Repository revision:")
!git -C {WORKTREE_ROOT} rev-parse HEAD

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")

print("Repository ready:", WORKTREE_ROOT)


### 0.5 Install Python Dependencies

What this step does: runs pip from the repository checkout and installs the project Colab requirements file.

Required inputs: WORKTREE_ROOT containing requirements-colab.txt and Python package index access.

Produced outputs: the notebook runtime has the project Colab Python dependencies installed.

When this step may be skipped: only when these exact requirements have already been installed in the active Colab runtime.

In [ ]:
%cd {WORKTREE_ROOT}

%pip install --upgrade pip

%pip install -r "requirements-colab.txt"

print("Repository requirements installed. Please restart the session if Colab asks for it.")


### 0.6 Add Source Tree To Python Path

What this step does: changes to the repository checkout and prepends the local src tree to sys.path for notebook imports.

Required inputs: WORKTREE_ROOT with a src directory from the checked-out repository.

Produced outputs: WORKTREE_ROOT / "src" is available on sys.path for subsequent imports.

When this step may be skipped: only when the same src path is already present on sys.path in this runtime.

In [ ]:
import sys

%cd {WORKTREE_ROOT}

source_root = WORKTREE_ROOT / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

print("Repository src directory added to sys.path:", source_root)


### 0.7 Import Base Workflow API And Build Runtime Plan

What this step does: imports only the Base workflow API and builds the workflow-owned runtime plan that describes restore specs, resolved run policy, and mirrored publish destination.

Required inputs: the installed dependencies, sys.path entry from step 0.6, WORKTREE_ROOT, DRIVE_PROJECT_ROOT, RUN_MODE, RUN_NAME, BASE_OUTPUT_POLICY, and Drive-published processed artifacts for every required split.

Produced outputs: base_runtime_plan, base_workflow_config, and workflow functions used by later cells; no project core, data, artifacts, modeling, or visualization modules are imported by the notebook.

When this step may be skipped: only when base_runtime_plan already reflects the current roots, run controls, Drive artifacts, and output policy.


In [ ]:
from text_to_sign_production.workflows.base import (
    build_base_publish_plan,
    build_base_runtime_plan,
    build_base_runtime_readiness_summary,
    run_base_workflow,
    summarize_base_publish_plan,
    summarize_base_workflow_outputs,
    validate_base_inputs,
    verify_base_runtime_inputs,
    write_base_checkpoint_manifest,
    write_base_prediction_sample_archive_member_list,
    write_base_qualitative_sample_archive_member_list,
)

base_runtime_plan = build_base_runtime_plan(
    project_root=WORKTREE_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    run_name=RUN_NAME,
    run_mode=RUN_MODE,
    output_policy=BASE_OUTPUT_POLICY,
)
base_workflow_config = base_runtime_plan.workflow_config

print("Base workflow API ready:", build_base_runtime_plan.__module__)
print("Run mode:", base_runtime_plan.run_mode)
print("Run name:", base_runtime_plan.run_name)
print("Train split:", base_runtime_plan.train_split)
print("Validation split:", base_runtime_plan.validation_split)
print("Prediction splits:", base_runtime_plan.prediction_splits)
print("Required splits:", base_runtime_plan.required_splits)
print("Run root:", base_runtime_plan.run_layout.run_root)
print("Expected publish run root:", base_runtime_plan.expected_publish_run_root)

# --- Readiness summary ---
base_readiness = build_base_runtime_readiness_summary(base_runtime_plan)

print()
print("Base runtime readiness summary")
print("==============================")
print("Live log path:", base_readiness.live_log_path)
print()
print("Recovery")
print("--------")
print("Recovery persistence enabled:", base_readiness.recovery_persistence_enabled)
print("Recovery root:", base_readiness.recovery_root)
print("Recovery manifest path:", base_readiness.recovery_manifest_path)
print()
print("Training / validation surface")
print("-----------------------------")
print("Training surface:", base_readiness.training_surface)
print("Validation surface:", base_readiness.validation_surface)
print()
print("Quality tier config")
print("-------------------")
print("Quality tier config path:", base_readiness.quality_tier_config_path)
print("Quality tier config sha256:", base_readiness.quality_tier_config_hash)
if base_readiness.tier_counts:
    print("Tier counts:")
    for split, counts in base_readiness.tier_counts.items():
        rendered = ", ".join(f"{tier}={n}" for tier, n in counts.items())
        print(f"  {split}: {rendered}")
else:
    print("Tier counts: not available")
print()
print("Primary metric")
print("--------------")
print("Primary metric name:", base_readiness.primary_metric_name)
print()
print("Target standardization")
print("----------------------")
print("Enabled:", base_readiness.target_standardization_enabled)
print("Fit surface:", base_readiness.target_standardization_fit_surface)
print("Leakage policy:", base_readiness.target_standardization_leakage_policy)
print("Metadata path:", base_readiness.target_standardization_metadata_path)
print("Inverse prediction transform:", base_readiness.target_standardization_inverse_prediction)
print()
print("Training policy")
print("---------------")
print("Max epochs:", base_readiness.max_epochs)
print("Min epochs:", base_readiness.min_epochs)
print("Early stopping patience:", base_readiness.early_stopping_patience)
print("Progress expectation:", base_readiness.progress_expectation)
print("Output policy:", base_readiness.output_policy)
print("Resume supported:", base_readiness.resume_supported)
print("Duration estimate:", base_readiness.duration_estimate)
print()
if base_readiness.check_mode_warning:
    print("Check mode warning:", base_readiness.check_mode_warning)
if base_readiness.complete_mode_warning:
    print("Complete mode warning:", base_readiness.complete_mode_warning)
print()
print("Risk-control reminders")
print("----------------------")
for reminder in base_readiness.risk_control_reminders:
    print(f"- {reminder}")


## 1. Restore Processed Base Inputs
### 1.1 Locate Required Processed Inputs

What this step does: reads the workflow restore specs and prints the required Drive sources, runtime targets, existence checks, and expected byte sizes.

Required inputs: base_runtime_plan from step 0.7.

Produced outputs: a human-readable restore plan for manifests and processed sample archives without executing restore commands.

When this step may be skipped: only when the same restore specs have already been inspected for the current runtime plan.

In [ ]:
print("Required splits:", base_runtime_plan.required_splits)

print("Processed manifest restore specs:")
for split, spec in base_runtime_plan.processed_manifest_files.items():
    print(f"- {split} source Drive manifest path:", spec.source_path)
    print(f"- {split} target runtime manifest path:", spec.target_path)
    print(f"- {split} source exists:", spec.source_path.is_file())
    print(f"- {split} expected bytes:", spec.expected_input_bytes)

print("Processed sample archive restore specs:")
for split, spec in base_runtime_plan.processed_sample_archives.items():
    print(f"- {split} source processed sample archive path:", spec.archive_path)
    print(f"- {split} target runtime sample split root:", spec.expected_split_root)
    print(f"- {split} source exists:", spec.archive_path.is_file())
    print(f"- {split} expected bytes:", spec.expected_input_bytes)


### 1.2 Copy Required Processed Manifests

What this step does: executes each workflow-provided processed manifest restore command with visible byte progress.

Required inputs: base_runtime_plan.processed_manifest_files and writable runtime data directories.

Produced outputs: processed manifest files restored into the runtime checkout.

When this step may be skipped: only when the same manifest files have already been restored into this checkout and will be verified in step 1.4.

In [ ]:
%cd {WORKTREE_ROOT}

for split, manifest_spec in base_runtime_plan.processed_manifest_files.items():
    spec = manifest_spec.restore_command
    print(f"Restoring {split} processed manifest:", spec.label)
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)


### 1.3 Extract Required Processed Sample Archives

What this step does: executes each workflow-provided processed sample archive extraction command with visible byte progress.

Required inputs: base_runtime_plan.processed_sample_archives, zstd support, and writable runtime processed sample directories.

Produced outputs: processed sample split roots restored for every split required by the Base workflow.

When this step may be skipped: only when the same processed sample split roots already exist in this checkout and will be verified in step 1.4.

In [ ]:
%cd {WORKTREE_ROOT}

for split, archive_spec in base_runtime_plan.processed_sample_archives.items():
    spec = archive_spec.extract_command
    print(f"Extracting {split} processed samples:", spec.label)
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)


### 1.4 Verify Restored Base Inputs

What this step does: asks the workflow verifier to check restored manifests, sample split roots, sample presence, and a small manifest-referenced sample subset.

Required inputs: base_runtime_plan after the manifest restore and processed sample extraction commands have run.

Produced outputs: base_input_summary with per-split manifest paths, manifest record counts, sample roots, sample counts, and checked manifest sample counts.

When this step may be skipped: only when base_input_summary has already been verified after the current restore operations.

In [ ]:
base_input_summary = verify_base_runtime_inputs(base_runtime_plan)
print("Verified required splits:", base_input_summary.required_splits)
for split, split_summary in base_input_summary.split_inputs.items():
    print("Split:", split)
    print("Manifest path:", split_summary.manifest_path)
    print("Manifest record count:", split_summary.manifest_record_count)
    print("Samples dir:", split_summary.samples_dir)
    print("Sample count:", split_summary.sample_count)
    print("Checked manifest sample count:", split_summary.checked_manifest_sample_count)


## 2. Run Base Workflow
### 2.1 Validate Base Workflow Inputs

What this step does: validates the workflow-owned Base configuration against the restored runtime inputs before training starts.

Required inputs: base_runtime_plan.workflow_config and verified processed inputs from step 1.4.

Produced outputs: validated Base workflow inputs plus printed run name, output policy, resolved splits, and resolved limits.

When this step may be skipped: only when the same workflow configuration has already been validated after the current restore operations.

In [ ]:
base_workflow_config = base_runtime_plan.workflow_config
validate_base_inputs(base_workflow_config)

print("Validation passed for run mode:", base_runtime_plan.run_mode)
print("Run name:", base_runtime_plan.run_name)
print("Output policy:", base_workflow_config.output_policy)
print("Train split:", base_runtime_plan.train_split)
print("Validation split:", base_runtime_plan.validation_split)
print("Prediction splits:", base_runtime_plan.prediction_splits)
print("Required splits:", base_runtime_plan.required_splits)
if base_runtime_plan.run_mode == "check":
    print("Run mode warning: This is a full-data check run, not final M0 baseline training.")
elif base_runtime_plan.run_mode == "complete":
    print("Run mode warning: This is the final M0 comparison-floor baseline training mode. It is not a sign-intelligibility or contribution-strength claim.")


### 2.2 Run M0 Base Workflow

What this step does: runs the workflow-owned M0 Base training, prediction export, reports, qualitative export, and output verification.

Required inputs: base_workflow_config from step 2.1 and verified processed runtime inputs from section 1.

Produced outputs: base_result from the completed Base workflow.

When this step may be skipped: only when base_result already comes from a completed run with the same workflow configuration.

In [ ]:
base_result = run_base_workflow(base_workflow_config)
print("Base workflow complete:", base_result.run_root)


### 2.3 Inspect Base Workflow Outputs

What this step does: summarizes the verified workflow outputs using notebook-safe workflow summary objects.

Required inputs: base_result from step 2.2.

Produced outputs: base_output_summary with run paths, checkpoint paths, prediction manifests, reports, qualitative artifacts when present, and verification status.

When this step may be skipped: only when base_output_summary already reflects the current base_result.

In [ ]:
base_output_summary = summarize_base_workflow_outputs(base_result)

print("Run root:", base_output_summary.run_root)
print("Config snapshot:", base_output_summary.config_snapshot_path)
print("Baseline config copy:", base_output_summary.baseline_config_copy_path)
print("Metrics path:", base_output_summary.training_metrics_path)
print("Training summary path:", base_output_summary.training_summary_path)
print("Run summary path:", base_output_summary.run_summary_path)
print("Best checkpoint path:", base_output_summary.best_checkpoint_path)
print("Last checkpoint path:", base_output_summary.last_checkpoint_path)
for split, split_summary in base_output_summary.prediction_splits.items():
    print(f"{split} prediction manifest path:", split_summary.manifest_path)
    print(f"{split} prediction manifest records:", split_summary.manifest_record_count)
    print(f"{split} prediction sample count:", split_summary.sample_count)
print("Baseline report JSON path:", base_output_summary.baseline_report_json_path)
print("Baseline report Markdown path:", base_output_summary.baseline_report_markdown_path)
print("Failure-mode report JSON path:", base_output_summary.failure_modes_json_path)
print("Failure-mode report Markdown path:", base_output_summary.failure_modes_markdown_path)
if base_output_summary.qualitative is None:
    print("Qualitative summary: not present")
else:
    print("Qualitative output dir:", base_output_summary.qualitative.output_dir)
    print("Qualitative panel definition path:", base_output_summary.qualitative.panel_definition_path)
    print("Qualitative records path:", base_output_summary.qualitative.records_path)
    print("Qualitative summary path:", base_output_summary.qualitative.panel_summary_path)
    print("Qualitative evidence path:", base_output_summary.qualitative.evidence_bundle_path)
print("Verification status:", base_output_summary.verification_status)


## 3. Publish Base Run Outputs
### 3.1 Build Base Publish Plan

What this step does: asks the workflow to build direct file copy specs, checkpoint compression specs, prediction sample archive specs, and the optional qualitative sample archive spec without executing commands.

Required inputs: base_result from step 2.2, WORKTREE_ROOT, and DRIVE_PROJECT_ROOT.

Produced outputs: base_publish_plan with mirrored runtime and Drive run roots plus workflow-generated publish command specs.

When this step may be skipped: only when base_publish_plan already reflects the current completed run and Drive project root.


In [ ]:
base_publish_plan = build_base_publish_plan(
    base_result,
    drive_project_root=DRIVE_PROJECT_ROOT,
    project_root=WORKTREE_ROOT,
)

print("Runtime run root:", base_publish_plan.runtime_run_root)
print("Drive run root:", base_publish_plan.drive_run_root)
print("Direct file specs:", len(base_publish_plan.direct_file_specs))
print("Checkpoint specs:", len(base_publish_plan.checkpoint_specs))
print("Prediction sample archive specs:", len(base_publish_plan.prediction_sample_archive_specs))
print("Qualitative sample archive:", base_publish_plan.qualitative_sample_archive_spec is not None)
for file_spec in base_publish_plan.direct_file_specs:
    print("Direct file:", file_spec.label, "->", file_spec.target_path)
for checkpoint_spec in base_publish_plan.checkpoint_specs:
    print("Checkpoint:", checkpoint_spec.role, "->", checkpoint_spec.target_path)
for archive_spec in base_publish_plan.prediction_sample_archive_specs:
    print("Prediction samples:", archive_spec.split, archive_spec.expected_member_count)
if base_publish_plan.qualitative_sample_archive_spec is None:
    print("No qualitative sample archive spec for this run.")
else:
    print("Qualitative samples:", base_publish_plan.qualitative_sample_archive_spec.expected_member_count)


### 3.2 Copy Direct Base Files

What this step does: executes only the workflow-generated byte-progress copy commands for small inspectable Base files.

Required inputs: base_publish_plan.direct_file_specs and writable Drive mirrored run directories.

Produced outputs: direct-published config, training metadata, prediction manifests, reports, qualitative metadata when present, and run summary files in Drive.

When this step may be skipped: only when these exact direct files have already been copied from the current runtime run and will be verified in step 3.8.


In [ ]:
for file_spec in base_publish_plan.direct_file_specs:
    spec = file_spec.copy_command
    print("Publishing direct file:", file_spec.label)
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)


### 3.3 Publish Base Checkpoints

What this step does: executes only workflow-generated checkpoint compression commands, writes the checkpoint manifest from verified compressed outputs, and copies that manifest with a workflow-generated command.

Required inputs: base_publish_plan.checkpoint_specs, zstd support, and writable Drive checkpoint directory.

Produced outputs: best.pt.zst, last.pt.zst, and checkpoint-manifest.json in the mirrored Drive run checkpoint directory.

When this step may be skipped: only when these exact checkpoint publish outputs already exist and will be verified in step 3.8.


In [ ]:
for checkpoint_spec in base_publish_plan.checkpoint_specs:
    spec = checkpoint_spec.compress_command
    print("Publishing checkpoint:", checkpoint_spec.role)
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

checkpoint_manifest_file_spec = write_base_checkpoint_manifest(base_publish_plan)
spec = checkpoint_manifest_file_spec.copy_command
print("Publishing checkpoint manifest:", checkpoint_manifest_file_spec.target_path)
print(spec.bash)
!bash -o pipefail -c {spec.arg}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(spec.failure)


### 3.4 Build Prediction Sample Archive Member Lists

What this step does: writes workflow-derived expected member lists for each prediction split sample archive.

Required inputs: base_publish_plan.prediction_sample_archive_specs and runtime prediction manifests.

Produced outputs: expected member list files consumed by the workflow-generated prediction sample archive commands.

When this step may be skipped: only when the same member lists have already been written from the current publish plan.


In [ ]:
for archive_spec in base_publish_plan.prediction_sample_archive_specs:
    member_list_path = write_base_prediction_sample_archive_member_list(archive_spec)
    print("Prediction member list:", archive_spec.split, member_list_path)
    print("Expected members:", archive_spec.expected_member_count)


### 3.5 Archive Base Prediction Samples

What this step does: executes only the workflow-generated per-split prediction sample archive commands with visible file-count progress.

Required inputs: prediction sample member lists from step 3.4, zstd support, and writable Drive prediction directories.

Produced outputs: samples.tar.zst for each prediction split in the mirrored Drive run tree.

When this step may be skipped: only when the same prediction sample archives already exist and will be verified in step 3.8.


In [ ]:
for archive_spec in base_publish_plan.prediction_sample_archive_specs:
    spec = archive_spec.archive_command
    print("Archiving prediction samples:", archive_spec.split)
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)


### 3.6 Build Qualitative Sample Archive Member List

What this step does: writes the workflow-derived expected member list for qualitative reference and prediction sample evidence when qualitative export is present.

Required inputs: base_publish_plan.qualitative_sample_archive_spec when qualitative export produced sample evidence.

Produced outputs: an expected qualitative sample member list consumed by the workflow-generated qualitative sample archive command, or a printed skip reason when qualitative export is absent.

When this step may be skipped: only when qualitative export is absent or the same member list has already been written from the current publish plan.


In [ ]:
qualitative_archive_spec = base_publish_plan.qualitative_sample_archive_spec
if qualitative_archive_spec is None:
    print("No qualitative sample archive to prepare for this run.")
else:
    member_list_path = write_base_qualitative_sample_archive_member_list(qualitative_archive_spec)
    print("Qualitative member list:", member_list_path)
    print("Expected members:", qualitative_archive_spec.expected_member_count)


### 3.7 Archive Base Qualitative Samples

What this step does: executes only the workflow-generated qualitative sample archive command when qualitative sample evidence exists.

Required inputs: qualitative sample member list from step 3.6, zstd support, and writable Drive qualitative directory when qualitative export is present.

Produced outputs: panel_samples.tar.zst in the mirrored Drive qualitative directory, or a printed skip reason when qualitative export is absent.

When this step may be skipped: only when qualitative export is absent or the same qualitative sample archive already exists and will be verified in step 3.8.


In [ ]:
qualitative_archive_spec = base_publish_plan.qualitative_sample_archive_spec
if qualitative_archive_spec is None:
    print("No qualitative sample archive to publish for this run.")
else:
    spec = qualitative_archive_spec.archive_command
    print("Archiving qualitative samples")
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)


### 3.8 Verify Published Base Outputs

What this step does: executes only workflow-generated archive listing commands, then asks the workflow to verify direct file copies, compressed checkpoints, checkpoint manifest metadata, archive member equality, and absence of legacy Base publish outputs.

Required inputs: published direct files, checkpoint outputs, prediction sample archives, optional qualitative sample archive, and base_publish_plan.

Produced outputs: base_publish_summary with verified direct file, checkpoint, prediction archive, qualitative archive, and overall publish status details.

When this step may be skipped: only when base_publish_summary already verifies the current publish plan after all publish commands have run.


In [ ]:
for archive_spec in base_publish_plan.prediction_sample_archive_specs:
    spec = archive_spec.list_command
    print("Listing prediction sample archive:", archive_spec.split)
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

qualitative_archive_spec = base_publish_plan.qualitative_sample_archive_spec
if qualitative_archive_spec is None:
    print("No qualitative sample archive to verify for this run.")
else:
    spec = qualitative_archive_spec.list_command
    print("Listing qualitative sample archive")
    print(spec.bash)
    !bash -o pipefail -c {spec.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(spec.failure)

base_publish_summary = summarize_base_publish_plan(base_publish_plan)
print("Publish verification status:", base_publish_summary.verification_status)
print("Direct files verified:", len(base_publish_summary.direct_files))
print("Checkpoints verified:", len(base_publish_summary.checkpoints))
print("Prediction sample archives verified:", len(base_publish_summary.prediction_archives))
print("Qualitative sample archive verified:", base_publish_summary.qualitative_archive is not None)
print("Missing count:", base_publish_summary.missing_count)
print("Invalid count:", base_publish_summary.invalid_count)


### 3.9 Inspect Final Published Base Outputs

What this step does: prints a closure-oriented final summary and the risk-control reminder without parsing report files in the notebook.

Required inputs: base_output_summary from step 2.3 and base_publish_summary from step 3.8.

Produced outputs: final runtime root, published Drive root, verified direct file count, published report paths, prediction manifest paths, checkpoint paths, qualitative paths when present, and risk-control reminder.

When this step may be skipped: only when this exact final summary has already been printed for the current run and publish plan.


In [ ]:
published_direct_files = {
    (summary.group, summary.split, summary.label): summary.target_path
    for summary in base_publish_summary.direct_files
}

print("Run mode:", base_runtime_plan.run_mode)
if base_runtime_plan.run_mode == "check":
    print("Run mode warning: This is a full-data check run, not final M0 baseline training.")
elif base_runtime_plan.run_mode == "complete":
    print("Run mode warning: This is the final M0 comparison-floor baseline training mode. It is not a sign-intelligibility or contribution-strength claim.")
def published_direct_file(label, group, split=None):
    key = (group, split, label)
    if key not in published_direct_files:
        raise KeyError(f"Published direct file was not verified: {key}")
    return published_direct_files[key]

print("Runtime run root:", base_publish_plan.runtime_run_root)
print("Published Drive run root:", base_publish_plan.drive_run_root)
print("Published direct files verified:", len(base_publish_summary.direct_files))
print("Published checkpoint manifest:", base_publish_summary.checkpoint_manifest.target_path)
print("Published reports:")
print("- Baseline JSON:", published_direct_file("baseline report JSON", "reports"))
print("- Baseline Markdown:", published_direct_file("baseline report Markdown", "reports"))
print("- Failure-mode JSON:", published_direct_file("failure-mode report JSON", "reports"))
print("- Failure-mode Markdown:", published_direct_file("failure-mode report Markdown", "reports"))
print("Published prediction manifests:")
for split in base_output_summary.prediction_splits:
    print(f"- {split}:", published_direct_file(f"{split} prediction manifest", "predictions", split))
print("Published checkpoints:")
for checkpoint_summary in base_publish_summary.checkpoints:
    print(f"- {checkpoint_summary.role}:", checkpoint_summary.target_path)
print("Published prediction sample archives:")
for archive_summary in base_publish_summary.prediction_archives:
    print(f"- {archive_summary.label}:", archive_summary.target_archive_path)
print("Published qualitative artifacts:")
if base_output_summary.qualitative is None:
    print("- qualitative: not present")
else:
    print("- records:", published_direct_file("qualitative records", "qualitative"))
    print("- panel summary:", published_direct_file("qualitative panel summary", "qualitative"))
    print("- evidence bundle:", published_direct_file("qualitative evidence bundle", "qualitative"))
    if base_publish_summary.qualitative_archive is not None:
        print("- sample archive:", base_publish_summary.qualitative_archive.target_archive_path)
print("Risk-control reminder: M0 is a comparison floor.")
print("Risk-control reminder: M0 is not a contribution-strength claim.")
print("Risk-control reminder: M0 is not a sign-intelligibility claim.")
